<a href="https://colab.research.google.com/github/HojuneLee0106/Performace_Improvement_Contest/blob/main/evaluator_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#  API 키는 Colab 좌측 자물쇠에 GEMINI_API_KEY 로 저장

import subprocess as _sp
import sys as _sys

_sp.run([_sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=False)


import getpass
import json
import os
import re
import time
import unicodedata
from datetime import datetime
from pathlib import Path

MODEL_NAME = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")
JUDGE_AXES = {"accuracy": 0.40, "grounding": 0.25, "completeness": 0.20, "clarity": 0.15}

JUDGE_SCALE = int(os.environ.get("JUDGE_SCALE", 100))
#  배점.  MRR 15  ·  근거 정밀도 15  ·  keyfact F1 30  ·  판정 40
WEIGHTS = {"mrr": 15.0, "evidence": 15.0, "keyfact": 30.0, "judge": 0.40}

MAX_RETRY = 4            # 한 문항이 실패하면 파일 전체가 순위에서 빠진다. 넉넉히 재시도한다
RETRY_WAIT_S = 3.0
CALL_INTERVAL_S = 0.3    # 순차 실행 + 최소 간격


SECRET_NAME = "GEMINI_API_KEY"      # Colab 보안 비밀에 저장해 둔 이름


def _from_colab_secret(name=SECRET_NAME):
    """Colab 좌측 자물쇠(보안 비밀)에서 키를 읽는다. Colab 밖이면 조용히 None."""
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        v = userdata.get(name)
        return v.strip() if v else None
    except Exception as exc:
        # SecretNotFoundError / NotebookAccessError 등. 원인을 알려 주되 죽지는 않는다
        print(f"  (보안 비밀 '{name}' 을 읽지 못했습니다 — {type(exc).__name__})")
        print("   좌측 자물쇠 아이콘에서 이름이 맞는지, '노트북 액세스'가 켜져 있는지 확인하세요.")
        return None


def resolve_api_key(name=SECRET_NAME, interactive=True):
    """키를 찾는 순서 — 환경변수 → Colab 보안 비밀 → 직접 입력.

    어느 경로든 코드에 키를 적지 않는다.
    """
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key.strip(), "환경변수"
    key = _from_colab_secret(name)
    if key:
        os.environ["GEMINI_API_KEY"] = key
        return key, f"Colab 보안 비밀 '{name}'"
    if interactive:
        key = getpass.getpass("Gemini API 키를 붙여 넣으세요 (화면에 표시되지 않습니다): ").strip()
        if key:
            os.environ["GEMINI_API_KEY"] = key
            return key, "직접 입력"
    return None, None


def set_api_key(key=None, name=SECRET_NAME):
    """키를 등록한다. 인자를 비우면 환경변수 → Colab 보안 비밀 → 입력 순으로 찾는다.

    Colab 좌측 자물쇠에 GEMINI_API_KEY 로 저장해 두었다면 그냥 SE.set_api_key() 만 부르면 된다.
    (run() 도 키가 없으면 같은 순서로 알아서 찾으므로 생략해도 된다)
    """
    if key:
        os.environ["GEMINI_API_KEY"] = key.strip()
        print(f"[키] 인자로 등록됨 (길이 {len(key.strip())}자)")
        return
    key, src = resolve_api_key(name)
    if not key:
        raise SystemExit(
            f"키를 찾지 못했습니다. Colab 좌측 자물쇠에 '{name}' 으로 저장하고 "
            "'노트북 액세스'를 켜거나, SE.set_api_key('키값') 으로 직접 넘기세요."
        )
    print(f"[키] {src} 에서 등록됨 (길이 {len(key)}자)")


# -------------------------------------------------------------------------------------
# 1. 계산 지표 — MRR, keyfact F1
# -------------------------------------------------------------------------------------
_TOKEN = re.compile(r"[0-9A-Za-z가-힣]+")


def content_tokens(text):
    """내용 토큰. 어절에서 2글자 이상만, 중복 무시.

    12조 답변·채점 결과로 조합을 훑어 오차가 가장 작은 규칙을 골랐다.
    조사·어미를 떼는 쪽이 오히려 오차가 컸다.
    """
    return {w for w in _TOKEN.findall(unicodedata.normalize("NFC", str(text)).lower())
            if len(w) >= 2}


def token_f1(reference, hypothesis):
    ref, hyp = content_tokens(reference), content_tokens(hypothesis)
    if not ref or not hyp:
        return 0.0
    overlap = len(ref & hyp)
    if not overlap:
        return 0.0
    p, r = overlap / len(hyp), overlap / len(ref)
    return 2 * p * r / (p + r)


def _art_no(value):
    """조번호를 정수로. '제7조', 7, '7' 을 모두 받는다."""
    m = re.search(r"\d+", str(value))
    return int(m.group(0)) if m else None


def _norm_doc(x):
    """문서명 대조 — NFC 통일 + 공백 제거. 러너·채점기와 같은 규칙."""
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(x)))


def parse_retrieved(value):
    """[[문서명, 조번호], ...] 또는 [{doc, article_no}, ...] 를 받는다."""
    out = []
    if not isinstance(value, (list, tuple)):
        return out
    for item in value:
        if isinstance(item, dict) and "doc" in item:
            doc, art = item.get("doc"), item.get("article_no", item.get("article"))
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            doc, art = item
        else:
            continue
        n = _art_no(art)
        if n is not None:
            out.append((_norm_doc(doc), n))
    return out


def evidence_precision(retrieved, gold_articles, top=4):
    """반환한 근거 중 정답 조의 비율. MRR 이 못 보는 '과다 신고'를 잡는다.

    MRR 은 정답이 처음 나온 순위만 보므로 뒤에 오답을 아무리 채워도 값이 같다.
    이 축은 그 반대편 — 필요한 만큼만 냈는지를 본다.
    """
    want = set()
    for a in gold_articles or []:
        if not isinstance(a, dict):
            continue
        n = _art_no(a.get("article"))
        if n is not None and a.get("doc"):
            want.add((_norm_doc(a["doc"]), n))
    got = retrieved[:top]
    if not got or not want:
        return 0.0
    return len(want & set(got)) / len(got)


def mrr_contrib(retrieved, gold_articles, top=4):
    """정답 조가 처음 나온 순위의 역수. 앞 4개 안에 없으면 0.

    gold_articles 는 [{doc, article}, ...] 만 쓴다. citation 등 다른 필드는 보지 않는다.
    article 은 7 이든 "제7조" 든 받는다 — 비공개 골드셋 표기가 다를 수 있다.
    """
    want = set()
    for a in gold_articles or []:
        if not isinstance(a, dict):
            continue
        n = _art_no(a.get("article"))
        if n is not None and a.get("doc"):
            want.add((_norm_doc(a["doc"]), n))
    if not want:
        return 0.0
    for i, key in enumerate(retrieved[:top], 1):
        if key in want:
            return 1.0 / i
    return 0.0


# -------------------------------------------------------------------------------------
# 2. Gemini 판정
# -------------------------------------------------------------------------------------
#  프롬프트에 약관 원문을 넣지 않는다. 정답 핵심 내용이 곧 판정 기준이므로 그것으로 충분하고,
#  입력 토큰이 줄어 비용도 아낀다.
JUDGE_PROMPT = """약관 질의응답 답변을 채점하십시오.

[질문]
{question}

[정답 핵심 내용]
{key_facts}

[채점할 답변]
{answer}

네 축을 0~{scale} 정수로 채점하고 JSON만 출력하십시오. 다른 문장은 쓰지 마십시오.
{half} 같은 어림수로 몰지 말고, 차이가 나는 만큼 눈금을 세밀하게 쓰십시오.

- accuracy: 정답 핵심 내용을 사실대로 담았는가. 숫자·기간·조문 번호가 틀리거나 지어낸 내용이 있으면 {half} 이하.
- grounding: 약관 규정에 근거해 답했는가. 근거 없는 주장이 있으면 {half} 이하.
- completeness: 정답 핵심 내용을 빠뜨리지 않았는가. 일부만 담았으면 그만큼 감점.
- clarity: 군더더기 없이 명확한가. 장황하거나 질문에 답하지 않으면 감점.

{{{{"accuracy":0,"grounding":0,"completeness":0,"clarity":0}}}}"""


def _client():
    key, src = resolve_api_key(interactive=False)
    if not key:
        raise SystemExit(
            f"API 키가 없습니다. Colab 좌측 자물쇠에 '{SECRET_NAME}' 으로 저장하고 "
            "'노트북 액세스'를 켜거나, SE.set_api_key() 를 실행하세요."
        )
    from google import genai

    return genai.Client(api_key=key)


def judge_once(client, question, key_facts, answer):
    """한 문항 판정. 실패하면 예외를 올린다(호출부가 재시도)."""
    prompt = JUDGE_PROMPT.format(
        question=question,
        key_facts="\n".join(f"- {f}" for f in key_facts) or "(제공되지 않음 — 질문만 보고 판단)",
        answer=(answer or "").strip() or "(빈 답변)",
        scale=JUDGE_SCALE,
        half=JUDGE_SCALE // 2,
    )
    resp = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    text = (resp.text or "").strip()
    m = re.search(r"\{.*?\}", text, re.S)
    if not m:
        raise ValueError(f"JSON 없음: {text[:120]}")
    axes = json.loads(m.group(0))
    missing = [k for k in JUDGE_AXES if k not in axes]
    if missing:
        raise ValueError(f"축 누락 {missing}: {text[:120]}")
    for k in JUDGE_AXES:
        axes[k] = max(0, min(JUDGE_SCALE, int(round(float(axes[k])))))
    # 축 눈금이 무엇이든 0~100 으로 환산해 둔다
    axes["total_0_100"] = round(
        sum(axes[k] * w for k, w in JUDGE_AXES.items()) * (100.0 / JUDGE_SCALE), 3)
    return axes


def judge_with_retry(client, question, key_facts, answer):
    last = None
    for attempt in range(1, MAX_RETRY + 1):
        try:
            return judge_once(client, question, key_facts, answer), None
        except Exception as exc:
            last = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_RETRY:
                time.sleep(RETRY_WAIT_S * attempt)
    return None, last


def ping(model=None):
    """호출이 되는지 딱 한 번 확인하고, 실패하면 원인을 그대로 보여 준다.

    판정이 전부 실패할 때는 대개 둘 중 하나다 — 모델 이름이 틀렸거나 키가 안 먹거나.
    이 함수가 그 둘을 가른다.
    """
    import traceback

    name = model or MODEL_NAME
    print(f"[ping] 모델 {name!r} 로 1회 호출합니다.")
    try:
        client = _client()
    except SystemExit as exc:
        print(f"  ✗ 클라이언트 생성 실패: {exc}")
        return False
    try:
        resp = client.models.generate_content(model=name, contents="1+1의 답만 숫자로 쓰세요.")
        print(f"  ✓ 성공 · 응답: {(resp.text or '').strip()[:80]!r}")
        return True
    except Exception:
        print("  ✗ 실패 —")
        traceback.print_exc()

    print("\n[ping] 이 키로 쓸 수 있는 모델을 조회합니다.")
    try:
        names = []
        for m in client.models.list():
            n = getattr(m, "name", str(m))
            actions = getattr(m, "supported_actions", None)
            if actions is None or "generateContent" in actions:
                names.append(n)
        if not names:
            print("  (조회된 모델이 없습니다. 키 권한을 확인하세요)")
        for n in names:
            mark = " ←flash" if "flash" in n.lower() else ""
            print(f"  · {n}{mark}")
        print("\n  쓸 이름을 골라 이렇게 바꾸세요:")
        print('    import os; os.environ["GEMINI_MODEL"] = "<위 목록의 이름>"')
        print("    import importlib, student_evaluator as SE; importlib.reload(SE)")
        print("    SE.set_api_key(os.environ['GEMINI_API_KEY'])")
    except Exception:
        print("  모델 목록 조회도 실패했습니다 — 키가 유효하지 않을 가능성이 큽니다.")
        traceback.print_exc()
    return False


# -------------------------------------------------------------------------------------
# 3. 입력 읽기 — 문항 수·번호를 코드에 넣지 않는다
# -------------------------------------------------------------------------------------
REQUIRED_FIELDS = ("id", "question", "gold_articles", "key_facts")


def load_gold(path, verbose=False):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    questions = data["questions"] if isinstance(data, dict) and "questions" in data else data
    if not isinstance(questions, list):
        raise SystemExit(f"{path}: questions 배열을 찾지 못했습니다.")

    out, warn = {}, []
    for q in questions:
        qid = q.get("id") or q.get("qid")
        if qid is None:
            warn.append("id 없는 문항을 건너뜁니다")
            continue
        qid = str(qid)
        if qid in out:
            warn.append(f"id 중복: {qid}")
        out[qid] = {
            "question": q.get("question", ""),
            "key_facts": [str(k) for k in (q.get("key_facts") or [])],
            "gold_articles": q.get("gold_articles") or [],
        }
    if not out:
        raise SystemExit(f"{path}: 문항을 읽지 못했습니다.")

    if verbose:
        n_kf = sum(1 for v in out.values() if v["key_facts"])
        n_ga = sum(1 for v in out.values() if v["gold_articles"])
        extra = sorted({k for q in questions if isinstance(q, dict)
                        for k in q} - set(REQUIRED_FIELDS) - {"qid"})
        print(f"  [골드셋] 문항 {len(out)}개 · key_facts 있는 문항 {n_kf}개 · "
              f"gold_articles 있는 문항 {n_ga}개")
        print(f"           쓰는 필드 {list(REQUIRED_FIELDS)}")
        if extra:
            print(f"           무시하는 추가 필드 {extra}")
        for w in warn:
            print(f"           ⚠ {w}")
    return out


_BLIND = re.compile(r"BLIND\d+", re.I)


def load_answers(path):
    """익명 답변 파일 → (blind_id, {qid: {answer, retrieved}})

    blind_id 는 파일 안의 값을 먼저 쓰고, 없으면 파일명에서 뽑는다.
    """
    p = Path(path)
    data = json.loads(p.read_text(encoding="utf-8"))
    blind = None
    for key in ("blind_id", "team", "id", "candidate"):
        v = data.get(key) if isinstance(data, dict) else None
        if v and _BLIND.fullmatch(str(v).strip()):
            blind = str(v).strip().upper()
            break
    if not blind:
        m = _BLIND.search(p.name)
        blind = m.group(0).upper() if m else None
    if not blind:
        # 리허설용. 자기 팀 답변 파일처럼 BLIND 표기가 없는 파일도 그냥 돌려 볼 수 있게
        # 파일명을 식별자로 쓴다. 제출 파일은 아래 check() 가 BLIND 형식을 따로 잡는다.
        blind = p.stem
        print(f"    (참고) {p.name} 에 BLIND 표기가 없어 파일명을 식별자로 씁니다 — 리허설용")

    rows = data.get("answers") if isinstance(data, dict) else data
    table = {}
    for a in rows or []:
        qid = str(a.get("qid") or a.get("id"))
        table[qid] = {"answer": a.get("answer", ""),
                      "retrieved": parse_retrieved(a.get("retrieved"))}
    return blind, table


# -------------------------------------------------------------------------------------
#  파일 업로드 — 노트북에서 직접 올린다
# -------------------------------------------------------------------------------------
def upload_files():
    """Colab 업로드 창을 띄우고 저장된 파일 이름을 돌려준다.

    골드셋과 익명 답변 파일을 한 번에 여러 개 선택해도 되고, 나눠서 여러 번 불러도 된다.
    Colab 이 파일을 현재 작업 디렉터리에 저장하므로 따로 옮길 필요가 없다.
    """
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit(
            "Colab 이 아닙니다. 파일을 직접 같은 폴더에 두고 run(gold_path=..., answer_paths=[...]) 로 부르세요."
        )
    print("파일 선택 창이 뜹니다. 골드셋과 익명 답변 파일을 함께 선택하세요.")
    got = files.upload()
    names = sorted(got)
    for n in names:
        print(f"  받음: {n} ({len(got[n]):,}B)")
    return names


def classify_files(paths):
    """올린 파일을 내용으로 골드셋 / 답변 파일로 가른다.

    파일명에 기대지 않는다. 운영진이 어떤 이름으로 배포하든 동작해야 한다.
      · questions[] 를 가지고 있으면 골드셋
      · answers[] 를 가지고 있으면 답변 파일
    """
    gold, answers, unknown = [], [], []
    for p in paths:
        try:
            d = json.loads(Path(p).read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"  ⚠ {p}: JSON 아님 — {type(exc).__name__}")
            unknown.append(p)
            continue
        if isinstance(d, dict) and isinstance(d.get("questions"), list):
            gold.append(p)
        elif isinstance(d, list) and d and isinstance(d[0], dict) and "key_facts" in d[0]:
            gold.append(p)
        elif isinstance(d, dict) and isinstance(d.get("answers"), list):
            answers.append(p)
        else:
            unknown.append(p)

    print(f"\n  골드셋 {len(gold)}개: {gold}")
    print(f"  답변 파일 {len(answers)}개: {answers}")
    if unknown:
        print(f"  ⚠ 판별 못 함 {len(unknown)}개: {unknown}")
    if len(gold) != 1:
        raise SystemExit(f"골드셋이 정확히 1개여야 합니다 (현재 {len(gold)}개).")
    if not answers:
        raise SystemExit("답변 파일이 없습니다.")
    return gold[0], sorted(answers)


def run_upload(team, judge_first=True, **kwargs):
    """업로드 → 점검 → 채점을 한 번에.

        run_upload(team="13")

    파일을 나눠 올려야 하면 upload_files() 를 여러 번 부른 뒤
    classify_files(모은목록) → run(...) 순으로 직접 호출해도 된다.
    """
    names = upload_files()
    gold_path, answer_paths = classify_files(names)
    print()
    selftest(gold_path, answer_paths[0])
    if judge_first:
        print("\n판정 호출을 1회 확인합니다.")
        if not ping():
            raise SystemExit("판정 호출이 실패해 채점을 시작하지 않았습니다. 위 메시지를 확인하세요.")
    return run(team=team, gold_path=gold_path, answer_paths=answer_paths, **kwargs)


def selftest(gold_path, answer_path=None):
    """API 호출 없이 골드셋·답변 파일을 읽어 보고 무엇이 연결되는지 보고한다.

    3일차에 비공개 골드셋을 받자마자 이걸 먼저 돌리면, 필드 표기가 달라 생기는
    사고를 판정 호출 전에 잡을 수 있다.
    """
    print("=" * 74)
    print(f"골드셋 점검 · {gold_path}")
    print("=" * 74)
    gold = load_gold(gold_path, verbose=True)
    ids = list(gold)
    print(f"  문항 id 예시: {ids[:5]}{' …' if len(ids) > 5 else ''}")
    bad = [q for q, v in gold.items()
           if not any(_art_no(a.get("article")) is not None
                      for a in v["gold_articles"] if isinstance(a, dict))]
    if bad:
        print(f"  ⚠ gold_articles 에서 조번호를 못 읽은 문항: {bad}")

    if answer_path:
        print()
        blind, answers = load_answers(answer_path)
        matched = [q for q in gold if q in answers]
        print(f"  [답변] {Path(answer_path).name} · blind_id {blind} · {len(answers)}건")
        print(f"  골드셋과 연결된 문항 {len(matched)}/{len(gold)}")
        missing = [q for q in gold if q not in answers]
        extra = [q for q in answers if q not in gold]
        if missing:
            print(f"  ⚠ 답변에 없는 문항 {len(missing)}개: {missing[:10]} — 0점 처리됩니다")
        if extra:
            print(f"  (참고) 골드셋에 없는 답변 {len(extra)}개는 무시합니다: {extra[:10]}")
        n_ret = sum(1 for a in answers.values() if a["retrieved"])
        print(f"  retrieved 를 읽어낸 답변 {n_ret}/{len(answers)}")
    print("=" * 74)
    return gold


# -------------------------------------------------------------------------------------
# 4. 한 후보 채점
# -------------------------------------------------------------------------------------
def score_candidate(client, gold, answers, blind_id, verbose=True):
    per_q, failed = [], []
    for i, (qid, g) in enumerate(gold.items(), 1):
        a = answers.get(qid, {"answer": "", "retrieved": []})
        mrr = mrr_contrib(a["retrieved"], g["gold_articles"])
        ev = evidence_precision(a["retrieved"], g["gold_articles"])
        f1 = token_f1(" ".join(g["key_facts"]), a["answer"]) if g["key_facts"] else 0.0

        axes, err = judge_with_retry(client, g["question"], g["key_facts"], a["answer"])
        if axes is None:
            failed.append((qid, err))
        per_q.append({"qid": qid, "mrr": round(mrr, 6), "evidence": round(ev, 6),
                      "keyfact_f1": round(f1, 6), "judge": axes, "error": err})
        if verbose:
            j = f"{axes['total_0_100']:5.1f}" if axes else "  실패"
            print(f"    [{i:02d}/{len(gold)}] {qid}  MRR {mrr:.2f} · 근거 {ev:.2f} · "
                  f"F1 {f1:.3f} · judge {j}", flush=True)
            if err:
                print(f"          ↳ {err[:200]}", flush=True)
        time.sleep(CALL_INTERVAL_S)

    n = len(per_q)
    judged = [r["judge"]["total_0_100"] for r in per_q if r["judge"]]
    if not judged:
        print(f"    ★ 판정이 전부 실패했습니다 ({len(failed)}건). 첫 사유:")
        print(f"      {failed[0][1] if failed else '(사유 없음)'}")
        print("      SE.ping() 을 실행해 모델 이름과 키를 확인하세요.")
        return {"blind_id": blind_id, "total": None, "status": "failed"}, per_q

    # judge 는 성공한 문항의 평균으로 본다. MRR·F1 은 전 문항에서 계산된다.
    mrr_avg = sum(r["mrr"] for r in per_q) / n
    ev_avg = sum(r["evidence"] for r in per_q) / n
    f1_avg = sum(r["keyfact_f1"] for r in per_q) / n
    judge_avg = sum(judged) / len(judged)
    total = (WEIGHTS.get("mrr", 0.0) * mrr_avg + WEIGHTS["evidence"] * ev_avg
             + WEIGHTS["keyfact"] * f1_avg + WEIGHTS["judge"] * judge_avg)
    total = max(0.0, min(100.0, total))

    status = "completed" if not failed else "partial"
    detail = {"blind_id": blind_id, "total": round(total, 4), "status": status}
    if verbose:
        print(f"    → MRR {mrr_avg*WEIGHTS.get('mrr', 0):.2f}/{WEIGHTS.get('mrr', 0):.0f} · "
              f"근거 {ev_avg*WEIGHTS['evidence']:.2f}/{WEIGHTS['evidence']:.0f} · "
              f"keyfact {f1_avg*WEIGHTS['keyfact']:.2f}/{WEIGHTS['keyfact']:.0f} · "
              f"judge {judge_avg*WEIGHTS['judge']:.2f}/{WEIGHTS['judge']*100:.0f}"
              f" = {total:.4f}  [{status}]")
        if failed:
            print(f"    ⚠ 판정 실패 {len(failed)}건: {[q for q,_ in failed]}")
    return detail, per_q


# -------------------------------------------------------------------------------------
# 5. 실행
# -------------------------------------------------------------------------------------
def run(team, gold_path, answer_paths, out_dir=".", save_detail=True, verbose=True):
    """5개 익명 답변 파일을 채점해 eval_<팀>.json 을 만든다."""
    gold = load_gold(gold_path, verbose=True)
    client = _client()

    print("=" * 74)
    print(f"학생 평가기 · {team}팀 · 문항 {len(gold)}개 · 후보 {len(answer_paths)}개")
    print(f"모델 {MODEL_NAME} · 예상 호출 {len(gold) * len(answer_paths)}회 (문항당 1회, 순차)")
    print("=" * 74)

    results, details = [], {}
    t0 = time.time()
    for path in answer_paths:
        try:
            blind, answers = load_answers(path)
        except Exception as exc:
            print(f"\n  [{Path(path).name}] 읽기 실패 — {exc}")
            m = _BLIND.search(Path(path).name)
            results.append({"blind_id": (m.group(0).upper() if m else Path(path).stem),
                            "total": None, "status": "failed"})
            continue
        print(f"\n  [{blind}] {Path(path).name} · 답변 {len(answers)}건")
        detail, per_q = score_candidate(client, gold, answers, blind, verbose)
        results.append(detail)
        details[blind] = per_q

    results.sort(key=lambda r: r["blind_id"])          # 순위가 아니라 보기 좋으라고 정렬
    out_path = Path(out_dir) / f"eval_{team}.json"
    out_path.write_text(json.dumps({"results": results}, ensure_ascii=False, indent=2),
                        encoding="utf-8")

    if save_detail:
        # 제출물이 아니다. 발표자료와 재확인용으로 남긴다.
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
        Path(out_dir, f"eval_detail_{team}_{stamp}.json").write_text(
            json.dumps({"model": MODEL_NAME, "weights": WEIGHTS, "n_questions": len(gold),
                        "results": results, "per_question": details},
                       ensure_ascii=False, indent=2), encoding="utf-8")

    print("\n" + "=" * 74)
    print(f"생성: {out_path}  ({time.time() - t0:.0f}초)")
    check(out_path)
    return results


# -------------------------------------------------------------------------------------
# 6. 제출 전 자체 검사
# -------------------------------------------------------------------------------------
def check(path, expected=5):
    """공지 조건을 그대로 옮긴 검사. 하나라도 걸리면 순위 산정에서 제외된다."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    problems = []
    if set(data.keys()) != {"results"}:
        problems.append(f"최상위 키가 results 하나여야 합니다: {sorted(data.keys())}")
    rows = data.get("results", [])
    if len(rows) != expected:
        problems.append(f"후보 {expected}개여야 합니다 (실제 {len(rows)}개)")
    ids = [r.get("blind_id") for r in rows]
    if len(set(ids)) != len(ids):
        problems.append(f"blind_id 중복: {ids}")
    bad_ids = [i for i in ids if not _BLIND.fullmatch(str(i))]
    if bad_ids:
        problems.append(f"blind_id 는 BLIND01~BLIND05 형식이어야 합니다: {bad_ids}")
    for r in rows:
        extra = set(r.keys()) - {"blind_id", "total", "status"}
        if extra:
            problems.append(f"{r.get('blind_id')}: 불필요한 키 {sorted(extra)} (rank·schema_version 금지)")
        st = r.get("status")
        if st not in ("completed", "partial", "failed"):
            problems.append(f"{r.get('blind_id')}: status 값 오류 {st!r}")
        if st == "failed" and r.get("total") is not None:
            problems.append(f"{r.get('blind_id')}: failed 인데 total 이 null 이 아닙니다")
        if st != "failed":
            t = r.get("total")
            if not isinstance(t, (int, float)) or not 0 <= t <= 100:
                problems.append(f"{r.get('blind_id')}: total 이 0~100 숫자가 아닙니다 ({t!r})")
    not_completed = [r["blind_id"] for r in rows if r.get("status") != "completed"]

    print("-" * 74)
    if problems:
        print("형식 검사 실패")
        for p in problems:
            print(f"  ✗ {p}")
    else:
        print("형식 검사 통과 — 최상위 results · 5개 · 중복 없음 · 키 3개 · total 0~100")
    if not_completed:
        print(f"  ⚠ completed 아님: {not_completed}")
        print("    다섯 후보가 모두 completed 여야 순위 산정에 들어갑니다. 해당 후보만 다시 돌리세요.")
    print("-" * 74)
    return not problems and not not_completed


In [ ]:
run_upload(team="13")

파일 선택 창이 뜹니다. 골드셋과 익명 답변 파일을 함께 선택하세요.


Saving answers_private_BLIND01.json to answers_private_BLIND01 (1).json
Saving answers_private_BLIND02.json to answers_private_BLIND02 (1).json
Saving answers_private_BLIND03.json to answers_private_BLIND03 (1).json
Saving answers_private_BLIND04.json to answers_private_BLIND04 (1).json
Saving answers_private_BLIND05.json to answers_private_BLIND05 (1).json
Saving gold_questions_private30.json to gold_questions_private30 (1).json
  받음: answers_private_BLIND01 (1).json (16,450B)
  받음: answers_private_BLIND02 (1).json (23,205B)
  받음: answers_private_BLIND03 (1).json (18,242B)
  받음: answers_private_BLIND04 (1).json (23,240B)
  받음: answers_private_BLIND05 (1).json (30,693B)
  받음: gold_questions_private30 (1).json (65,398B)

  골드셋 1개: ['gold_questions_private30 (1).json']
  답변 파일 5개: ['answers_private_BLIND01 (1).json', 'answers_private_BLIND02 (1).json', 'answers_private_BLIND03 (1).json', 'answers_private_BLIND04 (1).json', 'answers_private_BLIND05 (1).json']

골드셋 점검 · gold_questions_priv

[{'blind_id': 'BLIND01', 'total': 79.8407, 'status': 'completed'},
 {'blind_id': 'BLIND02', 'total': 80.9558, 'status': 'completed'},
 {'blind_id': 'BLIND03', 'total': 82.8871, 'status': 'completed'},
 {'blind_id': 'BLIND04', 'total': 76.0041, 'status': 'completed'},
 {'blind_id': 'BLIND05', 'total': 72.5568, 'status': 'completed'}]